In [ ]:
# ============================================================
# BLOCK 0: CONFIGURATION
# Edit paths here. Nothing else in the notebook needs changing.
# ============================================================

from pathlib import Path

# ── Project directory ──────────────────────────────────────────
try:
    PROJECT_DIR = Path(__file__).resolve().parent
except NameError:
    PROJECT_DIR = Path().resolve()

# ── User-configurable paths ────────────────────────────────────
# Mendeley ECG image dataset (folder containing Normal/, MI/, etc.)
# Download: https://data.mendeley.com/datasets/gwbz3fsgp8/2
MENDELEY_DATASET_PATH = Path("path/to/mendeley-ecg-dataset")

# Where ptbxl_generate.py stores generated PNG images (match OUTPUT_DIR in that script)
PTBXL_IMAGES_DIR = Path("path/to/ptbxl-generated-images")

# Weights file — trained on Mendeley; loaded or overwritten depending on RETRAIN_MODEL
WEIGHTS_PATH = PROJECT_DIR / "mendeley_model_weights.pth"

# PTB-XL manifest written by ptbxl_generate.py (must be beside this notebook)
PTBXL_MANIFEST = PROJECT_DIR / "ptbxl_manifest.csv"

# All generated figures go here (created automatically)
FIGURES_DIR = PROJECT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# All numerical results (JSON, CSV) go here (created automatically)
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Set False to load existing weights and skip training.
# Set True to train from scratch and overwrite mendeley_model_weights.pth.
RETRAIN_MODEL = False

# ── Sanity checks ──────────────────────────────────────────────
import sys

if not MENDELEY_DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Mendeley dataset not found:\n  {MENDELEY_DATASET_PATH}\n"
        "Update MENDELEY_DATASET_PATH in BLOCK 0."
    )

if not RETRAIN_MODEL and not WEIGHTS_PATH.exists():
    raise FileNotFoundError(
        f"Model weights not found:\n  {WEIGHTS_PATH}\n"
        "Either set RETRAIN_MODEL = True to train the model, "
        "or place mendeley_model_weights.pth in the project folder."
    )

print(f"Project folder   : {PROJECT_DIR}")
print(f"Mendeley dataset : {MENDELEY_DATASET_PATH}")
print(f"Weights file     : {WEIGHTS_PATH}")
print(f"PTB-XL manifest  : {PTBXL_MANIFEST}")
print(f"Figures folder   : {FIGURES_DIR}")
print(f"Results folder   : {RESULTS_DIR}")
print(f"Retrain model    : {RETRAIN_MODEL}")

In [ ]:
# ============================================================
# BLOCK 1: Imports and Reproducibility
# ============================================================

import json
import os
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
import ast
import random
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, Subset, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, matthews_corrcoef,
    balanced_accuracy_score, f1_score,
)
from scipy.signal import medfilt, find_peaks
from skimage.morphology import skeletonize
import copy

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Seed set to: {seed}")

set_seed(42)

CLASS_NAMES = ['Normal', 'MI', 'Abnormal', 'History_MI']
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")
print(f"Dataset path: {MENDELEY_DATASET_PATH}")

In [ ]:
# ============================================================
# BLOCK 2: 1D-ResNet Model Architecture
# Enhanced version: 4 residual blocks (64→128→256→512)
# with dropout for regularisation
# ============================================================

class ResNetBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.1):
        super().__init__()
        self.conv1    = nn.Conv1d(in_channels, out_channels, kernel_size=5,
                                  stride=stride, padding=2, bias=False)
        self.bn1      = nn.BatchNorm1d(out_channels)
        self.conv2    = nn.Conv1d(out_channels, out_channels, kernel_size=5,
                                  padding=2, bias=False)
        self.bn2      = nn.BatchNorm1d(out_channels)
        self.dropout  = nn.Dropout(dropout)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm1d(out_channels))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class ECGNet(nn.Module):
    """
    4-block 1D-ResNet for paper ECG classification.
    Input:  (batch, 12, 1000) — 12 leads, 1000 samples each
    Output: (batch, 4)        — logits for 4 diagnostic classes
    """
    def __init__(self, num_classes=4):
        super().__init__()
        self.conv1   = nn.Conv1d(12, 64, kernel_size=7, stride=2,
                                  padding=3, bias=False)
        self.bn1     = nn.BatchNorm1d(64)
        self.layer1  = ResNetBlock1D(64,  64)
        self.layer2  = ResNetBlock1D(64,  128, stride=2)
        self.layer3  = ResNetBlock1D(128, 256, stride=2)
        self.layer4  = ResNetBlock1D(256, 512, stride=2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc      = nn.Linear(512, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self.fc(self.dropout(self.avgpool(x).view(x.size(0), -1)))

print("Model architecture defined.")
print(f"Parameters: {sum(p.numel() for p in ECGNet().parameters()):,}")

In [ ]:
# ============================================================
# BLOCK 3: Digitization Pipeline Functions
# SNR estimation, grid-aware calibration,
# Zhang-Suen skeletonization, lead extraction
# ============================================================

def calculate_snr(img):
    """
    Estimates SNR from the horizontal midrow of the image.
    Used for quality filtering (threshold 5.0 dB) and
    stratified evaluation (15 dB, 25 dB).
    """
    h, w  = img.shape
    row   = img[h//2, :w//4].astype(float)
    clean = medfilt(row, kernel_size=3)
    noise = row - clean
    return 10 * np.log10(np.mean(clean**2) / (np.mean(noise**2) + 1e-10))


def detect_grid_and_pulse(img):
    """
    Physically calibrated scale factor (mV per pixel).
    Two mechanisms:
      1. Grid projection — estimates pixels/mm from grid line spacing
      2. Contour pulse  — detects the 10mm calibration pulse rectangle
    Returns scale factor to convert pixel displacement to millivolts.
    """
    h, w = img.shape

    # Grid projection: find peaks in vertical projection of left quarter
    v_proj = np.mean(img[:, :w//4], axis=1)
    peaks, _ = find_peaks(-v_proj, distance=5)
    pixels_per_mm = (np.median(np.diff(peaks)) / 5.0
                     if len(peaks) > 1 else 5.6)

    # Contour pulse detection in leftmost 8% of image
    pulse_area = img[:, :int(w * 0.08)]
    _, thresh  = cv2.threshold(pulse_area, 0, 255,
                               cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        _, _, wc, hc = cv2.boundingRect(cnt)
        if hc > 40 and wc < 50:      # standard 10mm pulse proportions
            return 1.0 / hc          # 1mV / pulse_height_px

    # Fallback to grid projection
    return 1.0 / (pixels_per_mm * 10.0)


def extract_signal(image_path, augment=False):
    """
    Full pipeline: load → resize → SNR → calibrate → skeletonize → extract leads.
    Returns (signal_tensor [12,1000], snr_float).
    augment=True adds Gaussian noise + amplitude scaling (training only).
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return torch.zeros((12, 1000)), 0.0

    img   = cv2.resize(img, (1065, 560))
    h, w  = img.shape
    snr   = calculate_snr(img)
    scale = detect_grid_and_pulse(img)

    # Otsu binarization + Zhang-Suen skeletonization
    _, binary  = cv2.threshold(img, 0, 255,
                               cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    skeleton   = skeletonize(binary / 255).astype(np.uint8) * 255

    # Partition into 3x4 grid for 12-lead layout
    row_bounds = [0, int(h * 0.33), int(h * 0.66), h]
    all_leads  = []

    for i in range(3):
        row = skeleton[row_bounds[i]:row_bounds[i+1], :]
        lw  = w // 4
        for j in range(4):
            seg = row[:, j*lw:(j+1)*lw]
            raw = []
            for col in range(seg.shape[1]):
                idx = np.where(seg[:, col] > 0)[0]
                raw.append(np.mean(idx) if len(idx) > 0 else seg.shape[0]//2)
            raw = np.array(raw)

            # Calibrate to millivolts
            cal = (raw - np.mean(raw)) * scale

            # Optional signal augmentation (training only)
            if augment:
                cal = (cal + np.random.normal(0, 0.01, cal.shape)) * \
                      np.random.uniform(0.9, 1.1)

            resampled = cv2.resize(cal.reshape(1, -1), (1000, 1)).flatten()
            all_leads.append(resampled)

    return torch.tensor(np.array(all_leads), dtype=torch.float32), snr


print("Pipeline functions defined.")

In [ ]:
# ============================================================
# BLOCK 4: Dataset Class
# Loads Mendeley ECG images, applies SNR filter,
# reports per-class counts before/after filtering
# ============================================================

class MendeleyECGDataset(Dataset):
    CLASS_MAP = {
        "Normal":       0,
        "MI":           1,
        "Abnormal":     2,
        "History of MI": 3
    }

    def __init__(self, root_dir, snr_threshold=5.0, augment=False):
        self.augment       = augment
        self.snr_threshold = snr_threshold
        self.file_list     = []

        root_dir = Path(root_dir)
        temp_list = []
        for folder, label in self.CLASS_MAP.items():
            f_path = root_dir / folder
            if f_path.exists():
                for f in os.listdir(f_path):
                    if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                        temp_list.append((str(f_path / f), label))

        before = {i: 0 for i in range(4)}
        after  = {i: 0 for i in range(4)}
        print(f"Filtering {len(temp_list)} images (SNR \u2265 {snr_threshold} dB)...")

        for path, label in temp_list:
            before[label] += 1
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                snr = calculate_snr(cv2.resize(img, (1065, 560)))
                if snr >= snr_threshold:
                    self.file_list.append((path, label))
                    after[label] += 1

        inv = {v: k for k, v in self.CLASS_MAP.items()}
        print(f"\n{'Class':<22} {'Before':>7} {'After':>7} {'Rejected':>9}")
        print("-" * 48)
        for i in range(4):
            print(f"{inv[i]:<22} {before[i]:>7} {after[i]:>7}"
                  f" {before[i]-after[i]:>9}")
        print(f"{'TOTAL':<22} {sum(before.values()):>7}"
              f" {sum(after.values()):>7}"
              f" {sum(before.values())-sum(after.values()):>9}\n")

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        path, label = self.file_list[idx]
        sig, snr    = extract_signal(path, augment=self.augment)
        return sig, label, snr


train_dataset = MendeleyECGDataset(MENDELEY_DATASET_PATH, snr_threshold=5.0, augment=True)
test_dataset  = MendeleyECGDataset(MENDELEY_DATASET_PATH, snr_threshold=5.0, augment=False)


In [ ]:
# ============================================================
# BLOCK 5: Stratified Split and Class-Weighted Loss
# 80/20 stratified split, deterministic seed
# Class weights computed from training indices only
# ============================================================

set_seed(42)

train_idx, test_idx = train_test_split(
    range(len(train_dataset)),
    test_size=0.2,
    stratify=[x[1] for x in train_dataset.file_list],
    random_state=42
)

train_loader = DataLoader(
    Subset(train_dataset, train_idx), batch_size=16, shuffle=True)
test_loader  = DataLoader(
    Subset(test_dataset,  test_idx),  batch_size=16, shuffle=False)

n_train = len(train_idx)
n_test  = len(test_idx)
print(f"Train samples: {n_train}")
print(f"Test samples : {n_test}")

# Class weights: inverse frequency of training-split labels only
label_counts = np.array([
    sum(1 for i in train_idx if train_dataset.file_list[i][1] == cls)
    for cls in range(4)
])
class_weights = torch.tensor(
    1.0 / (label_counts / label_counts.sum()), dtype=torch.float32)
class_weights = (class_weights / class_weights.sum() * 4).to(DEVICE)
print(f"\nTrain class counts: {label_counts}")
print(f"Class weights: {class_weights.cpu().numpy().round(3)}")

# Save the exact train/test split for reproducibility
mendeley_split_df = pd.DataFrame({
    'file_path': [train_dataset.file_list[i][0] for i in range(len(train_dataset))],
    'label':     [train_dataset.file_list[i][1] for i in range(len(train_dataset))],
    'split':     ['train' if i in set(train_idx) else 'test'
                  for i in range(len(train_dataset))],
})
mendeley_split_df.to_csv(RESULTS_DIR / 'mendeley_split_seed42.csv', index=False)
print(f"\nSplit index saved: {RESULTS_DIR / 'mendeley_split_seed42.csv'}")

In [ ]:
# ============================================================
# BLOCK 6: Enhanced Pipeline Training
# Skipped when RETRAIN_MODEL = False (loads saved weights).
# 30 epochs, cosine annealing LR, class-weighted loss.
# ============================================================

set_seed(42)
model = ECGNet(num_classes=4).to(DEVICE)

if RETRAIN_MODEL:
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=30, eta_min=1e-5)

    print("--- Training Enhanced Pipeline (30 epochs) ---\n")
    for epoch in range(1, 31):
        model.train()
        epoch_loss = 0.0
        for sigs, labels, _ in train_loader:
            sigs, labels = sigs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(sigs), labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        print(f"Epoch {epoch:02d}/30 | "
              f"Loss: {epoch_loss/len(train_loader):.4f} | "
              f"LR: {scheduler.get_last_lr()[0]:.6f}")

    torch.save(model.state_dict(), WEIGHTS_PATH)
    print(f"\nWeights saved: {WEIGHTS_PATH}")
else:
    if not WEIGHTS_PATH.exists():
        raise FileNotFoundError(
            f"Cannot load weights — file not found:\n  {WEIGHTS_PATH}\n"
            "Set RETRAIN_MODEL = True in BLOCK 0 to train the model first."
        )
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
    model.eval()
    print(f"Loaded weights from: {WEIGHTS_PATH}")
    print("(Set RETRAIN_MODEL = True in BLOCK 0 to retrain.)")


In [ ]:
# ============================================================
# BLOCK 7: Full Evaluation on Mendeley Test Set
# Reports per-class metrics, MCC, balanced accuracy, macro F1,
# bootstrap 95% CIs, SNR stratification.
# Generates Figures 3 and 4 (saved to figures/)
# NOTE: The Mendeley dataset does not include patient IDs, so
# image-level independence cannot be guaranteed.
# ============================================================

def bootstrap_ci(y_true, y_pred, metric_fn, n=1000, seed=42):
    rng  = np.random.RandomState(seed)
    vals = []
    idx  = np.arange(len(y_true))
    for _ in range(n):
        s = rng.choice(idx, size=len(idx), replace=True)
        try:
            vals.append(metric_fn(y_true[s], y_pred[s]))
        except Exception:
            pass
    vals = np.array(vals)
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def evaluate_model(model, loader, label=None, save_prefix='cm'):
    model.eval()
    y_true, y_pred, all_snrs = [], [], []

    with torch.no_grad():
        for sigs, labels, snrs in loader:
            _, pred = torch.max(model(sigs.to(DEVICE)), 1)
            y_true.extend(labels.numpy())
            y_pred.extend(pred.cpu().numpy())
            all_snrs.extend(snrs.numpy())

    y_true  = np.array(y_true)
    y_pred  = np.array(y_pred)
    snr_arr = np.array(all_snrs)
    n_test  = len(y_true)
    label   = label or f'Mendeley — Full Test Set (N={n_test})'

    acc   = float(np.mean(y_true == y_pred))
    mcc   = float(matthews_corrcoef(y_true, y_pred))
    bacc  = float(balanced_accuracy_score(y_true, y_pred))
    mf1   = float(f1_score(y_true, y_pred, average='macro'))

    acc_lo,  acc_hi  = bootstrap_ci(y_true, y_pred,
        lambda a, b: np.mean(a == b))
    bacc_lo, bacc_hi = bootstrap_ci(y_true, y_pred,
        lambda a, b: balanced_accuracy_score(a, b))
    mf1_lo,  mf1_hi  = bootstrap_ci(y_true, y_pred,
        lambda a, b: f1_score(a, b, average='macro'))

    print(f"\n{'='*65}")
    print(f"  {label}")
    print(f"{'='*65}")
    print(f"  Accuracy : {acc*100:.2f}%  95% CI [{acc_lo*100:.2f}%, {acc_hi*100:.2f}%]")
    print(f"  Bal. Acc : {bacc*100:.2f}%  95% CI [{bacc_lo*100:.2f}%, {bacc_hi*100:.2f}%]")
    print(f"  Macro F1 : {mf1:.4f}   95% CI [{mf1_lo:.4f}, {mf1_hi:.4f}]")
    print(f"  MCC      : {mcc:.4f}")
    print()
    print(classification_report(y_true, y_pred,
                                target_names=CLASS_NAMES, digits=4))

    for thresh in [15, 25]:
        mask = snr_arr > thresh
        if mask.sum() == 0:
            continue
        a = np.mean(y_true[mask] == y_pred[mask])
        m = matthews_corrcoef(y_true[mask], y_pred[mask])
        print(f"  SNR > {thresh} dB (N={mask.sum()}) | "
              f"Acc: {a*100:.2f}%  MCC: {m:.4f}")

    cm = confusion_matrix(y_true, y_pred)
    out_path = FIGURES_DIR / f"{save_prefix}.png"
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'Confusion Matrix — {label}')
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f"  Saved: {out_path}")

    return (acc, mcc, bacc, mf1,
            (acc_lo, acc_hi), (bacc_lo, bacc_hi), (mf1_lo, mf1_hi),
            y_true, y_pred, snr_arr)


eval_out = evaluate_model(
    model, test_loader,
    label=f'Mendeley — Full Test Set (N={len(test_idx)})',
    save_prefix='cm_final_enhanced_pipeline')

(acc_all, mcc_all, bacc_all, mf1_all,
 acc_ci, bacc_ci, mf1_ci,
 yt_all, yp_all, snrs_all) = eval_out

# Save metrics
mendeley_metrics = {
    'n_test':           len(test_idx),
    'accuracy':         acc_all,
    'accuracy_ci95':    acc_ci,
    'balanced_accuracy': bacc_all,
    'balanced_accuracy_ci95': bacc_ci,
    'macro_f1':         mf1_all,
    'macro_f1_ci95':    mf1_ci,
    'mcc':              mcc_all,
    'note': (
        'The Mendeley dataset does not include patient identifiers; '
        'image-level independence is assumed but cannot be verified.'
    ),
}
with open(RESULTS_DIR / 'mendeley_metrics.json', 'w') as f:
    json.dump(mendeley_metrics, f, indent=2)
print(f"\nMetrics saved: {RESULTS_DIR / 'mendeley_metrics.json'}")

print("\n--- High Fidelity Subset (SNR > 25 dB) ---")
mask25 = snrs_all > 25
cm25   = confusion_matrix(yt_all[mask25], yp_all[mask25])
out25  = FIGURES_DIR / 'confusion_matrix_snr25.png'
plt.figure(figsize=(8, 6))
sns.heatmap(cm25, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix — High Fidelity (SNR > 25 dB)')
plt.tight_layout()
plt.savefig(out25, dpi=150)
plt.show()
print(f"Saved: {out25}")
acc25 = np.mean(yt_all[mask25] == yp_all[mask25]) * 100
mcc25 = matthews_corrcoef(yt_all[mask25], yp_all[mask25])
print(f"SNR>25 Accuracy: {acc25:.2f}%  MCC: {mcc25:.4f}  N={mask25.sum()}")

In [ ]:
# ============================================================
# BLOCK 8: Multi-component Pipeline Comparison
# Three extraction methods under identical model, split, seed:
#   Baseline A: Argmin + fixed scale (ECG_Model.py)
#   Baseline B: Canny pulse + skeletonization (ECG_Model_after.py)
#   Final:      Contour + grid + skeletonization (final pipeline)
# Generates Figure 5 (bar chart)
# ============================================================

def extract_argmin(image_path):
    """Baseline A: column-wise argmin, fixed 0.01 mV/px scale."""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return torch.zeros((12, 1000)), 0.0
    img   = cv2.resize(img, (1065, 560))
    h, w  = img.shape
    snr   = calculate_snr(img)
    scale = 0.01    # fixed scale — no physical calibration
    bounds = [0, int(h*0.33), int(h*0.66), h]
    leads  = []
    for i in range(3):
        row = img[bounds[i]:bounds[i+1], :]
        lw  = w // 4
        for j in range(4):
            seg = row[:, j*lw:(j+1)*lw]
            raw = np.argmin(seg, axis=0).astype(float)
            cal = (raw - np.mean(raw)) * scale
            leads.append(cv2.resize(cal.reshape(1,-1),(1000,1)).flatten())
    return torch.tensor(np.array(leads), dtype=torch.float32), snr


def extract_canny(image_path):
    """Baseline B: Canny edge pulse detection + skeletonization."""
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return torch.zeros((12, 1000)), 0.0
    img   = cv2.resize(img, (1065, 560))
    h, w  = img.shape
    snr   = calculate_snr(img)

    # Canny-based pulse height detection (unreliable on annotations)
    pulse_edges = cv2.Canny(img[:, :int(w*0.06)], 50, 150)
    positions   = np.where(pulse_edges > 0)[0]
    if len(positions) > 5:
        detected_height = np.max(positions) - np.min(positions)
        scale = 1.0 / detected_height if detected_height > 0 else 0.01
    else:
        scale = 0.01

    _, binary  = cv2.threshold(img, 0, 255,
                               cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    skeleton   = skeletonize(binary / 255).astype(np.uint8) * 255
    bounds     = [0, int(h*0.33), int(h*0.66), h]
    leads      = []
    for i in range(3):
        row = skeleton[bounds[i]:bounds[i+1], :]
        lw  = w // 4
        for j in range(4):
            seg = row[:, j*lw:(j+1)*lw]
            raw = []
            for col in range(seg.shape[1]):
                idx = np.where(seg[:, col] > 0)[0]
                raw.append(np.mean(idx) if len(idx) > 0 else seg.shape[0]//2)
            cal = (np.array(raw) - np.mean(raw)) * scale
            leads.append(cv2.resize(cal.reshape(1,-1),(1000,1)).flatten())
    return torch.tensor(np.array(leads), dtype=torch.float32), snr


def run_ablation(extract_fn, name):
    """Train and evaluate one pipeline variant from scratch."""
    set_seed(42)

    class AblationDataset(Dataset):
        def __init__(self, file_list):
            self.file_list = file_list
        def __len__(self): return len(self.file_list)
        def __getitem__(self, idx):
            path, label = self.file_list[idx]
            sig, snr    = extract_fn(path)
            return sig, label, snr

    full_files = test_dataset.file_list
    tr_files   = [full_files[i] for i in train_idx]
    te_files   = [full_files[i] for i in test_idx]

    tr_loader = DataLoader(AblationDataset(tr_files), batch_size=16, shuffle=True)
    te_loader = DataLoader(AblationDataset(te_files), batch_size=16, shuffle=False)

    set_seed(42)
    m     = ECGNet(num_classes=4).to(DEVICE)
    opt   = optim.Adam(m.parameters(), lr=0.001, weight_decay=1e-4)
    crit  = nn.CrossEntropyLoss(weight=class_weights)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30, eta_min=1e-5)

    for epoch in range(1, 31):
        m.train()
        for sigs, labels, _ in tr_loader:
            sigs, labels = sigs.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            crit(m(sigs), labels).backward()
            opt.step()
        sched.step()

    m.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for sigs, labels, _ in te_loader:
            _, pred = torch.max(m(sigs.to(DEVICE)), 1)
            y_true.extend(labels.numpy())
            y_pred.extend(pred.cpu().numpy())

    acc  = np.mean(np.array(y_true) == np.array(y_pred)) * 100
    mcc  = matthews_corrcoef(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred) * 100
    mf1  = f1_score(y_true, y_pred, average='macro')
    print(f"\n{name}: Acc={acc:.2f}%  Bal.Acc={bacc:.2f}%  MCC={mcc:.4f}")
    print(classification_report(y_true, y_pred,
                                target_names=CLASS_NAMES, digits=4))
    return acc, mcc


print("Running multi-component pipeline comparison (3 variants × 30 epochs each)...")
print("This will take approximately 20–30 minutes on CPU.\n")

acc_a, mcc_a = run_ablation(extract_argmin, "Baseline A: Argmin + fixed scale")
acc_b, mcc_b = run_ablation(extract_canny,  "Baseline B: Canny + skeletonization")
# Final pipeline already evaluated in Block 7
acc_f, mcc_f = acc_all * 100, mcc_all

# Summary table
print(f"\n{'='*65}")
print(f"  MULTI-COMPONENT PIPELINE COMPARISON")
print(f"{'='*65}")
print(f"  {'Method':<42} {'Accuracy':>10} {'MCC':>8}")
print(f"  {'-'*60}")
for name, acc, mcc in [
    ("Baseline A: Argmin + fixed scale",          acc_a, mcc_a),
    ("Baseline B: Canny pulse + skeletonization", acc_b, mcc_b),
    ("Final: Contour + grid + skeletonization",   acc_f, mcc_f),
]:
    print(f"  {name:<42} {acc:>9.2f}% {mcc:>8.4f}")
print(f"{'='*65}")

# Figure 5
methods = [
    "Baseline A:\nArgmin + fixed scale",
    "Baseline B:\nCanny pulse + skeleton",
    "Final:\nContour + grid + skeleton"
]
accs = [acc_a, acc_b, acc_f]
mccs = [mcc_a, mcc_b, mcc_f]
x    = np.arange(len(methods))
w    = 0.35

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.bar(x - w/2, accs, w, color='steelblue', label='Accuracy (%)')
ax2.bar(x + w/2, mccs, w, color='coral',     label='MCC')
ax1.set_xticks(x)
ax1.set_xticklabels(methods, ha='center')
ax1.set_ylabel('Accuracy (%)'); ax2.set_ylabel('MCC')
ax1.set_ylim(0, 105);           ax2.set_ylim(0, 1.1)
ax1.set_title('Multi-component Pipeline Comparison')
lines1, labs1 = ax1.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labs1+labs2, loc='upper left')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ablation_comparison.png', dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / 'ablation_comparison.png'}")

In [ ]:
# ============================================================
# BLOCK 9: Synthetic Degradation Experiment
# Applies rotation, blur, and brightness changes to test images
# Model held fixed — tests pipeline robustness
# Generates Figure 6 (degradation line plot)
# ============================================================

def degrade_and_evaluate(deg_fn, label, severity):
    """Apply degradation to each test image, run pipeline, evaluate."""
    y_true, y_pred = [], []
    model.eval()

    with torch.no_grad():
        for idx in test_idx:
            path, gt = test_dataset.file_list[idx]
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if img is None: continue
            img = cv2.resize(img, (1065, 560))
            img = deg_fn(img)          # apply degradation

            # Run pipeline on degraded image
            h, w  = img.shape
            scale = detect_grid_and_pulse(img)
            _, binary  = cv2.threshold(img, 0, 255,
                                       cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
            skeleton   = skeletonize(binary / 255).astype(np.uint8) * 255
            bounds     = [0, int(h*0.33), int(h*0.66), h]
            leads      = []
            for i in range(3):
                row = skeleton[bounds[i]:bounds[i+1], :]
                lw  = w // 4
                for j in range(4):
                    seg = row[:, j*lw:(j+1)*lw]
                    raw = []
                    for col in range(seg.shape[1]):
                        idx2 = np.where(seg[:, col] > 0)[0]
                        raw.append(np.mean(idx2) if len(idx2) > 0
                                   else seg.shape[0]//2)
                    cal = (np.array(raw) - np.mean(raw)) * scale
                    leads.append(cv2.resize(cal.reshape(1,-1),(1000,1)).flatten())

            sig  = torch.tensor(np.array(leads),
                                dtype=torch.float32).unsqueeze(0).to(DEVICE)
            pred = torch.argmax(model(sig), dim=1).item()
            y_true.append(gt)
            y_pred.append(pred)

    acc = np.mean(np.array(y_true) == np.array(y_pred)) * 100
    mcc = matthews_corrcoef(y_true, y_pred)
    print(f"  {label:<25} {severity:<15} Acc: {acc:.2f}%  MCC: {mcc:.4f}")
    return acc, mcc


# Degradation functions
def rotate(img, angle):
    h, w = img.shape
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_REPLICATE)

def blur(img, ksize):
    k = ksize if ksize % 2 == 1 else ksize + 1
    return cv2.GaussianBlur(img, (k, k), 0)

def brightness(img, factor):
    return np.clip(img.astype(float) * factor, 0, 255).astype(np.uint8)


# Baseline (no degradation)
print("Running degradation experiment...\n")
print(f"  {'Degradation':<25} {'Severity':<15} {'Accuracy':>10}  {'MCC':>8}")
print("  " + "-"*60)

base_acc = acc_all * 100
base_mcc = mcc_all
print(f"  {'No degradation':<25} {'—':<15} {base_acc:>9.2f}%  {base_mcc:>8.4f}")

deg_configs = [
    (lambda i: rotate(i,  5),       "Rotation",    "Mild (+5°)"),
    (lambda i: rotate(i, 15),       "Rotation",    "Medium (+15°)"),
    (lambda i: rotate(i, 30),       "Rotation",    "Severe (+30°)"),
    (lambda i: blur(i,  3),         "Blur",        "Mild (k=3)"),
    (lambda i: blur(i,  7),         "Blur",        "Medium (k=7)"),
    (lambda i: blur(i, 15),         "Blur",        "Severe (k=15)"),
    (lambda i: brightness(i, 1.2),  "Brightness",  "Mild (+20%)"),
    (lambda i: brightness(i, 1.4),  "Brightness",  "Medium (+40%)"),
    (lambda i: brightness(i, 0.6),  "Brightness",  "Severe (-40%)"),
]

deg_results = []
for fn, label, severity in deg_configs:
    a, m = degrade_and_evaluate(fn, label, severity)
    deg_results.append((label, severity, a, m))

# Figure 6: Degradation line plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
groups = [
    ("Rotation",   ["Mild\n(+5°)",  "Medium\n(+15°)", "Severe\n(+30°)"]),
    ("Blur",       ["Mild\n(k=3)",  "Medium\n(k=7)",  "Severe\n(k=15)"]),
    ("Brightness", ["Mild\n(+20%)", "Medium\n(+40%)", "Severe\n(-40%)"]),
]
for ax, (dtype, sevs) in zip(axes, groups):
    accs_d = [r[2] for r in deg_results if r[0] == dtype]
    ax.axhline(base_acc, color='gray', linestyle='--', linewidth=1.2,
               label=f'Baseline ({base_acc:.1f}%)')
    ax.plot(sevs, accs_d, marker='o', linewidth=2, color='steelblue')
    ax.set_title(dtype)
    ax.set_ylabel('Accuracy (%)')
    ax.set_ylim(0, 105)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Pipeline Robustness Under Synthetic Degradation', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'degradation_experiment.png', dpi=150)
plt.show()
print(f"\nSaved: {FIGURES_DIR / 'degradation_experiment.png'}")

In [ ]:
# ============================================================
# BLOCK 10: Cross-Dataset Domain Shift Analysis (PTB-XL)
# Requires ptbxl_manifest.csv from ptbxl_generate.py.
#
# Analysis A (zero-shot): E1-E6 preprocessing variants, all 600 images
# Analysis B (zero-shot): E1-E6 preprocessing variants, fold-10 images only
# E7 (domain-adapted):   train on folds 1-8, validate on fold 9 (early stop),
#                        evaluate once on fold 10 — never touched during training
#
# Creates 4 required result files in RESULTS_DIR.
# Generates Figures 7 and 8 in FIGURES_DIR.
# ============================================================

# ── Validate manifest ─────────────────────────────────────────
if not PTBXL_MANIFEST.exists():
    raise FileNotFoundError(
        f"PTB-XL manifest not found:\n  {PTBXL_MANIFEST}\n"
        "Run ptbxl_generate.py first."
    )

manifest = pd.read_csv(PTBXL_MANIFEST)

REQUIRED_COLS = {'image_path', 'label', 'strat_fold', 'patient_id',
                 'ecg_id', 'filename_hr'}
missing_cols = REQUIRED_COLS - set(manifest.columns)
if missing_cols:
    raise ValueError(
        f"Manifest missing columns: {missing_cols}\n"
        "Re-run ptbxl_generate.py."
    )

# Resolve relative image_path values using PTBXL_IMAGES_DIR
manifest['image_path'] = manifest['image_path'].apply(
    lambda p: str(PTBXL_IMAGES_DIR / p) if not Path(p).is_absolute() else p
)

missing_imgs = [r for r in manifest['image_path'] if not Path(r).exists()]
if missing_imgs:
    raise FileNotFoundError(
        f"{len(missing_imgs)} image(s) in manifest not found.\n"
        f"First: {missing_imgs[0]}\nRe-run ptbxl_generate.py."
    )

print(f"Manifest: {len(manifest)} images")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {(manifest['label']==i).sum()}")

# ── Fold-based splits ─────────────────────────────────────────
# Fold 10 is the test set and is NEVER evaluated during training or epoch selection.
train_manifest = manifest[manifest['strat_fold'].isin(range(1, 9))].copy()  # folds 1-8
val_manifest   = manifest[manifest['strat_fold'] == 9].copy()               # fold 9
test_manifest  = manifest[manifest['strat_fold'] == 10].copy()              # fold 10

print(f"\nFold split (PTB-XL stratified 10-fold):")
print(f"  Train  folds 1-8  : {len(train_manifest)} images")
print(f"  Val    fold  9    : {len(val_manifest)} images")
print(f"  Test   fold  10   : {len(test_manifest)} images")

# ── Three pairwise patient-overlap assertions ─────────────────
train_patients = set(train_manifest['patient_id'])
val_patients   = set(val_manifest['patient_id'])
test_patients  = set(test_manifest['patient_id'])

tv_overlap = train_patients & val_patients
vt_overlap = val_patients   & test_patients
tt_overlap  = train_patients & test_patients
assert not tv_overlap, \
    f"Train/val patient overlap: {len(tv_overlap)} patients"
assert not vt_overlap, \
    f"Val/test patient overlap: {len(vt_overlap)} patients"
assert not tt_overlap, \
    f"Train/test patient overlap: {len(tt_overlap)} patients"
print("\nPatient overlap assertions:")
print("  train ∩ val  = 0  OK")
print("  val  ∩ test  = 0  OK")
print("  train ∩ test = 0  OK")

# ── Preprocessing variants ────────────────────────────────────

def _remove_grid(binary):
    hk = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
    vk = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 40))
    return cv2.subtract(binary,
           cv2.add(cv2.morphologyEx(binary, cv2.MORPH_OPEN, hk),
                   cv2.morphologyEx(binary, cv2.MORPH_OPEN, vk)))

def _extract_leads_from_skel(skel, h, w, scale):
    bounds = [0, int(h*0.33), int(h*0.66), h]
    leads  = []
    for i in range(3):
        row = skel[bounds[i]:bounds[i+1], :]
        lw  = w // 4
        for j in range(4):
            seg = row[:, j*lw:(j+1)*lw]
            raw = [np.mean(np.where(seg[:,c]>0)[0])
                   if np.any(seg[:,c]>0) else seg.shape[0]//2
                   for c in range(seg.shape[1])]
            cal = (np.array(raw) - np.mean(raw)) * scale
            leads.append(cv2.resize(cal.reshape(1,-1),(1000,1)).flatten())
    return torch.tensor(np.array(leads), dtype=torch.float32)

# E1: Standard pipeline (identical to Mendeley training pipeline)
def ptbxl_e1(path):
    sig, _ = extract_signal(path)
    return sig

# E2: Grid removal + Otsu binarization + skeleton
def ptbxl_e2(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None: return torch.zeros((12,1000))
    img   = cv2.resize(img, (1065, 560))
    h, w  = img.shape
    scale = detect_grid_and_pulse(img)
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
    skel  = skeletonize(_remove_grid(binary)/255).astype(np.uint8)*255
    return _extract_leads_from_skel(skel, h, w, scale)

# E3: Min-max normalisation (cv2.NORM_MINMAX) + Otsu + skeleton — no grid removal
def ptbxl_e3(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None: return torch.zeros((12,1000))
    img   = cv2.resize(img, (1065, 560))
    img   = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)  # NOT histogram eq
    h, w  = img.shape
    scale = detect_grid_and_pulse(img)
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
    skel  = skeletonize(binary/255).astype(np.uint8)*255
    return _extract_leads_from_skel(skel, h, w, scale)

# E4: Adaptive Gaussian threshold + grid removal + skeleton
def ptbxl_e4(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None: return torch.zeros((12,1000))
    img    = cv2.resize(img, (1065, 560))
    h, w   = img.shape
    scale  = detect_grid_and_pulse(img)
    binary = cv2.adaptiveThreshold(img, 255,
                cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 8)
    skel   = skeletonize(_remove_grid(binary)/255).astype(np.uint8)*255
    return _extract_leads_from_skel(skel, h, w, scale)

# E5: Grid removal + Otsu + argmax (no skeleton)
def ptbxl_e5(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None: return torch.zeros((12,1000))
    img   = cv2.resize(img, (1065, 560))
    h, w  = img.shape
    scale = detect_grid_and_pulse(img)
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
    clean = _remove_grid(binary)
    bounds = [0, int(h*0.33), int(h*0.66), h]
    leads  = []
    for i in range(3):
        row = clean[bounds[i]:bounds[i+1], :]
        lw  = w // 4
        for j in range(4):
            seg = row[:, j*lw:(j+1)*lw]
            raw = np.argmax(seg, axis=0).astype(float)
            cal = (raw - np.mean(raw)) * scale
            leads.append(cv2.resize(cal.reshape(1,-1),(1000,1)).flatten())
    return torch.tensor(np.array(leads), dtype=torch.float32)

# E6: Min-max norm + grid removal + Otsu + skeleton
# Differs from E2 by the NORM_MINMAX step before binarization;
# differs from E3 by adding grid removal.
def ptbxl_e6(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None: return torch.zeros((12,1000))
    img   = cv2.resize(img, (1065, 560))
    img   = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)
    h, w  = img.shape
    scale = detect_grid_and_pulse(img)
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
    skel  = skeletonize(_remove_grid(binary)/255).astype(np.uint8)*255
    return _extract_leads_from_skel(skel, h, w, scale)

EXPERIMENTS = [
    ('E1', 'Standard pipeline',             ptbxl_e1),
    ('E2', 'Grid removal',                  ptbxl_e2),
    ('E3', 'Min-max norm',                  ptbxl_e3),
    ('E4', 'Adaptive threshold',            ptbxl_e4),
    ('E5', 'No skeleton (argmax)',          ptbxl_e5),
    ('E6', 'Min-max norm + grid removal',   ptbxl_e6),
]

# ── Bootstrap CI (4 metrics, seed 42, invalid resamples skipped) ──
def _bs_ci(yt, yp, n=1000, seed=42):
    rng  = np.random.RandomState(seed)
    vals = {m: [] for m in ('acc', 'bacc', 'mf1', 'mcc')}
    for _ in range(n):
        s  = rng.choice(len(yt), len(yt), replace=True)
        ys, ps = yt[s], yp[s]
        vals['acc'].append(float(np.mean(ys == ps)))
        vals['bacc'].append(float(balanced_accuracy_score(ys, ps)))
        vals['mf1'].append(float(f1_score(ys, ps, average='macro',
                                          zero_division=0)))
        try:
            vals['mcc'].append(float(matthews_corrcoef(ys, ps)))
        except Exception:
            pass
    ci = {}
    for m, v in vals.items():
        v = np.array(v)
        ci[m] = (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return ci

def _metrics_and_ci(yt, yp):
    acc  = float(np.mean(yt == yp))
    bacc = float(balanced_accuracy_score(yt, yp))
    mf1  = float(f1_score(yt, yp, average='macro', zero_division=0))
    mcc  = float(matthews_corrcoef(yt, yp))
    ci   = _bs_ci(yt, yp)
    return {'accuracy': acc, 'balanced_accuracy': bacc,
            'macro_f1': mf1, 'mcc': mcc, 'ci': ci}

# ── Zero-shot evaluation helper ───────────────────────────────
def run_zero_shot(extract_fn, rows_df, zs_model):
    zs_model.eval()
    y_true, y_pred = [], []
    for _, row in rows_df.iterrows():
        sig = extract_fn(row['image_path'])
        with torch.no_grad():
            pred = torch.argmax(
                zs_model(sig.unsqueeze(0).to(DEVICE)), dim=1).item()
        y_true.append(int(row['label']))
        y_pred.append(pred)
    return np.array(y_true), np.array(y_pred)

zs_model = ECGNet().to(DEVICE)
zs_model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
zs_model.eval()

# ── Analysis A: full cohort, zero-shot (E1-E6) ───────────────
print("\n" + "="*65)
print("  ANALYSIS A — Zero-shot, full cohort (N=600)")
print("="*65)

results_a = {}
for code, name, fn in EXPERIMENTS:
    print(f"  {code}: {name}...", end=' ', flush=True)
    yt, yp = run_zero_shot(fn, manifest, zs_model)
    m = _metrics_and_ci(yt, yp)
    results_a[code] = {'name': name, 'n': len(yt),
                       'y_true': yt.tolist(), 'y_pred': yp.tolist(), **m}
    print(f"Acc={m['accuracy']*100:.2f}%  "
          f"BalAcc={m['balanced_accuracy']*100:.2f}%  "
          f"MacroF1={m['macro_f1']:.3f}  MCC={m['mcc']:.4f}")

# ── Analysis B: fold-10 only, zero-shot (E1-E6) ──────────────
print("\n" + "="*65)
print(f"  ANALYSIS B — Zero-shot, fold-10 only (N={len(test_manifest)})")
print("="*65)

results_b = {}
for code, name, fn in EXPERIMENTS:
    print(f"  {code}: {name}...", end=' ', flush=True)
    yt, yp = run_zero_shot(fn, test_manifest, zs_model)
    m = _metrics_and_ci(yt, yp)
    results_b[code] = {'name': name, 'n': len(yt),
                       'y_true': yt.tolist(), 'y_pred': yp.tolist(), **m}
    print(f"Acc={m['accuracy']*100:.2f}%  "
          f"BalAcc={m['balanced_accuracy']*100:.2f}%  "
          f"MacroF1={m['macro_f1']:.3f}  MCC={m['mcc']:.4f}")

# ── E7: Domain-adapted ────────────────────────────────────────
# Train: folds 1-8 (gradient).  Val: fold 9 (loss only, early stop).
# Test:  fold 10 — evaluated ONCE after training, never during.
print("\n" + "="*65)
print("  E7 — Domain-adapted (train folds 1-8, val fold 9, test fold 10)")
print("="*65)

def _build_tensors(rows_df, extract_fn):
    sigs, labs = [], []
    for _, row in rows_df.iterrows():
        sigs.append(extract_fn(row['image_path']))
        labs.append(int(row['label']))
    return torch.stack(sigs), torch.tensor(labs)

print("  Building tensors (folds 1-8)...")
ft_sigs_tr, ft_labs_tr = _build_tensors(train_manifest, ptbxl_e6)
print("  Building tensors (fold 9)...")
ft_sigs_val, ft_labs_val = _build_tensors(val_manifest, ptbxl_e6)
print("  Building tensors (fold 10)...")
ft_sigs_te, ft_labs_te = _build_tensors(test_manifest, ptbxl_e6)

ft_model = ECGNet().to(DEVICE)
ft_model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
for name_p, param in ft_model.named_parameters():
    if 'layer4' not in name_p and 'fc' not in name_p:
        param.requires_grad = False

ft_train_loader = DataLoader(
    TensorDataset(ft_sigs_tr, ft_labs_tr), batch_size=16, shuffle=True)
ft_val_loader   = DataLoader(
    TensorDataset(ft_sigs_val, ft_labs_val), batch_size=16, shuffle=False)

ft_opt  = optim.Adam(
    filter(lambda p: p.requires_grad, ft_model.parameters()), lr=1e-4)
ft_crit = nn.CrossEntropyLoss()

N_EPOCHS_E7 = 20
PATIENCE_E7 = 5
best_val_loss = float('inf')
patience_ctr  = 0
best_state_e7 = None
history_rows  = []

print(f"  Fine-tuning layer4 + fc for up to {N_EPOCHS_E7} epochs "
      f"(early stop patience={PATIENCE_E7})...")
for epoch in range(1, N_EPOCHS_E7 + 1):
    # ─ Training step (folds 1-8 only) ─
    ft_model.train()
    tr_loss = 0.0
    for sigs_b, labs_b in ft_train_loader:
        ft_opt.zero_grad()
        loss = ft_crit(ft_model(sigs_b.to(DEVICE)), labs_b.to(DEVICE))
        loss.backward()
        ft_opt.step()
        tr_loss += loss.item()
    tr_loss /= len(ft_train_loader)

    # ─ Validation step (fold 9, no gradient) ─
    ft_model.eval()
    vl_loss = 0.0
    vl_true, vl_pred = [], []
    with torch.no_grad():
        for sigs_b, labs_b in ft_val_loader:
            out = ft_model(sigs_b.to(DEVICE))
            vl_loss += ft_crit(out, labs_b.to(DEVICE)).item()
            vl_pred.extend(torch.argmax(out, dim=1).cpu().numpy())
            vl_true.extend(labs_b.numpy())
    vl_loss /= len(ft_val_loader)
    vl_acc  = float(np.mean(np.array(vl_true) == np.array(vl_pred)))
    vl_bacc = float(balanced_accuracy_score(vl_true, vl_pred))
    vl_mf1  = float(f1_score(vl_true, vl_pred, average='macro', zero_division=0))

    history_rows.append({
        'epoch': epoch,
        'train_loss': round(tr_loss, 6),
        'val_loss':   round(vl_loss, 6),
        'val_accuracy': round(vl_acc, 4),
        'val_balanced_accuracy': round(vl_bacc, 4),
        'val_macro_f1': round(vl_mf1, 4),
    })
    print(f"    Epoch {epoch:02d}: tr={tr_loss:.4f}  vl={vl_loss:.4f}  "
          f"vl_acc={vl_acc*100:.2f}%")

    # Early stopping on fold-9 val loss
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_state_e7 = copy.deepcopy(ft_model.state_dict())
        patience_ctr  = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE_E7:
            print(f"    Early stop at epoch {epoch} (patience={PATIENCE_E7})")
            break

# Restore best checkpoint (selected on fold-9 val loss)
ft_model.load_state_dict(best_state_e7)

# Evaluate on fold 10 — the single, unrepeated test evaluation
ft_model.eval()
with torch.no_grad():
    preds_e7 = torch.argmax(
        ft_model(ft_sigs_te.to(DEVICE)), dim=1).cpu().numpy()
yt_e7 = ft_labs_te.numpy()

m_e7 = _metrics_and_ci(yt_e7, preds_e7)
acc_e7, bacc_e7, mf1_e7, mcc_e7 = (m_e7['accuracy'], m_e7['balanced_accuracy'],
                                     m_e7['macro_f1'], m_e7['mcc'])
e7_ci = m_e7['ci']
print(f"\n  E7 Test (fold 10): Acc={acc_e7*100:.2f}%  "
      f"BalAcc={bacc_e7*100:.2f}%  MacroF1={mf1_e7:.3f}  MCC={mcc_e7:.4f}")

# ── Save required result files ────────────────────────────────

# 1. ptbxl_zero_shot_full.csv — zero-shot metrics on full cohort (Analysis A)
rows_zs = []
for code, d in results_a.items():
    ci = d['ci']
    rows_zs.append({
        'experiment': code, 'name': d['name'], 'n': d['n'],
        'accuracy': d['accuracy'],
        'acc_ci_lo': ci['acc'][0], 'acc_ci_hi': ci['acc'][1],
        'balanced_accuracy': d['balanced_accuracy'],
        'bacc_ci_lo': ci['bacc'][0], 'bacc_ci_hi': ci['bacc'][1],
        'macro_f1': d['macro_f1'],
        'mf1_ci_lo': ci['mf1'][0], 'mf1_ci_hi': ci['mf1'][1],
        'mcc': d['mcc'],
        'mcc_ci_lo': ci['mcc'][0], 'mcc_ci_hi': ci['mcc'][1],
    })
pd.DataFrame(rows_zs).to_csv(RESULTS_DIR / 'ptbxl_zero_shot_full.csv', index=False)

# 2. ptbxl_common_test.csv — per-image predictions on fold-10
#    (Analysis B zero-shot E1-E6 + E7 domain-adapted, one row per image)
ct_df = test_manifest[['ecg_id', 'patient_id', 'strat_fold',
                         'label', 'class_name']].copy().reset_index(drop=True)
ct_df.rename(columns={'label': 'y_true'}, inplace=True)
for code, d in results_b.items():
    ct_df[f'{code}_pred'] = d['y_pred']
ct_df['E7_pred'] = preds_e7.tolist()
ct_df.to_csv(RESULTS_DIR / 'ptbxl_common_test.csv', index=False)

# 3. ptbxl_finetuning_history.csv — epoch-by-epoch training/val metrics
pd.DataFrame(history_rows).to_csv(
    RESULTS_DIR / 'ptbxl_finetuning_history.csv', index=False)

# 4. split_summary.json — fold counts, patient counts, overlap proofs
split_summary = {
    'train_folds': list(range(1, 9)),
    'val_fold':    9,
    'test_fold':   10,
    'n_train':     len(train_manifest),
    'n_val':       len(val_manifest),
    'n_test':      len(test_manifest),
    'n_train_patients': len(train_patients),
    'n_val_patients':   len(val_patients),
    'n_test_patients':  len(test_patients),
    'train_val_patient_overlap':  len(tv_overlap),
    'train_test_patient_overlap': len(tt_overlap),
    'val_test_patient_overlap':   len(vt_overlap),
    'class_distribution_train': {
        CLASS_NAMES[i]: int((train_manifest['label'] == i).sum())
        for i in range(4)},
    'class_distribution_val': {
        CLASS_NAMES[i]: int((val_manifest['label'] == i).sum())
        for i in range(4)},
    'class_distribution_test': {
        CLASS_NAMES[i]: int((test_manifest['label'] == i).sum())
        for i in range(4)},
}
with open(RESULTS_DIR / 'split_summary.json', 'w') as f:
    json.dump(split_summary, f, indent=2)

# Additional: Analysis B metrics (supplementary)
metrics_b_out = {k: {kk: vv for kk, vv in v.items()
                     if kk not in ('y_true', 'y_pred')}
                 for k, v in results_b.items()}
with open(RESULTS_DIR / 'ptbxl_analysis_b_metrics.json', 'w') as f:
    json.dump(metrics_b_out, f, indent=2)

# Additional: E7 metrics (supplementary)
e7_metrics_out = {
    'name': 'Domain-adapted (train folds 1-8, val fold 9)',
    'n_train': len(train_manifest), 'n_val': len(val_manifest),
    'n_test':  len(test_manifest),
    'epochs_run': len(history_rows),
    'accuracy': acc_e7, 'accuracy_ci95': list(e7_ci['acc']),
    'balanced_accuracy': bacc_e7, 'balanced_accuracy_ci95': list(e7_ci['bacc']),
    'macro_f1': mf1_e7, 'macro_f1_ci95': list(e7_ci['mf1']),
    'mcc': mcc_e7, 'mcc_ci95': list(e7_ci['mcc']),
    'frozen_layers': 'layer1, layer2, layer3',
    'trained_layers': 'layer4, fc', 'lr': 1e-4,
}
with open(RESULTS_DIR / 'ptbxl_e7_metrics.json', 'w') as f:
    json.dump(e7_metrics_out, f, indent=2)

print(f"\nResult files saved to {RESULTS_DIR}:")
for fname in ['ptbxl_zero_shot_full.csv', 'ptbxl_common_test.csv',
              'ptbxl_finetuning_history.csv', 'split_summary.json',
              'ptbxl_analysis_b_metrics.json', 'ptbxl_e7_metrics.json']:
    print(f"  {fname}")

# ── Figure 7: Analysis A bar chart ────────────────────────────
codes_a = list(results_a.keys())
accs_a  = [results_a[c]['accuracy']*100        for c in codes_a]
bacc_a  = [results_a[c]['balanced_accuracy']*100 for c in codes_a]
x = np.arange(len(codes_a)); w = 0.35

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()
ax1.bar(x - w/2, accs_a, w, color='steelblue', label='Accuracy (%)')
ax2.bar(x + w/2, bacc_a, w, color='coral',     label='Balanced Acc (%)')
ax1.set_xticks(x)
ax1.set_xticklabels(
    [f"{c}\n{results_a[c]['name']}" for c in codes_a], fontsize=8, ha='center')
ax1.set_ylabel('Accuracy (%)'); ax2.set_ylabel('Balanced Accuracy (%)')
ax1.set_ylim(0, 105);           ax2.set_ylim(0, 105)
ax1.set_title('Analysis A — Zero-shot Domain Shift (PTB-XL, full cohort N=600)')
l1, lb1 = ax1.get_legend_handles_labels()
l2, lb2 = ax2.get_legend_handles_labels()
ax1.legend(l1+l2, lb1+lb2, loc='upper right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'domain_shift_analysis_a.png', dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / 'domain_shift_analysis_a.png'}")

# ── Figure 8: Analysis B (fold-10) zero-shot vs E7 ────────────
codes_b  = list(results_b.keys())
accs_b   = [results_b[c]['accuracy']*100 for c in codes_b] + [acc_e7*100]
labels8  = codes_b + ['E7']
names8   = [results_b[c]['name'] for c in codes_b] + \
           ['Domain-adapted\n(folds 1-8 train\nfold 9 val)']
colors8  = ['steelblue'] * len(codes_b) + ['darkorange']
x8 = np.arange(len(labels8))

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x8, accs_b, color=colors8)
ax.set_xticks(x8)
ax.set_xticklabels([f"{l}\n{n}" for l, n in zip(labels8, names8)],
                   fontsize=7.5, ha='center')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 105)
ax.set_title(f'Analysis B — Fold-10 (N={len(test_manifest)}): '
             f'Zero-shot (E1-E6) vs Domain-adapted (E7)')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='steelblue', label='Zero-shot'),
                   Patch(color='darkorange', label='Domain-adapted (E7)')],
          loc='upper right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'domain_shift_analysis_b.png', dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / 'domain_shift_analysis_b.png'}")

# Confusion matrix: best fold-10 zero-shot
best_b_code = max(results_b, key=lambda c: results_b[c]['accuracy'])
best_b      = results_b[best_b_code]
cm_best_b   = confusion_matrix(best_b['y_true'], best_b['y_pred'])
plt.figure(figsize=(7, 6))
sns.heatmap(cm_best_b, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'Best Zero-shot: {best_b_code} — {best_b["name"]}\n'
          f'Fold 10, Acc={best_b["accuracy"]*100:.2f}%  MCC={best_b["mcc"]:.4f}')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cm_ptbxl_zero_shot_best.png', dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / 'cm_ptbxl_zero_shot_best.png'}")

# Confusion matrix: E7 (fold 10)
cm_e7 = confusion_matrix(yt_e7, preds_e7)
plt.figure(figsize=(7, 6))
sns.heatmap(cm_e7, annot=True, fmt='d', cmap='Oranges',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'Domain-adapted E7 — Fold 10\n'
          f'Acc={acc_e7*100:.2f}%  MCC={mcc_e7:.4f}')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cm_ptbxl_domain_adapted.png', dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / 'cm_ptbxl_domain_adapted.png'}")

In [ ]:
# ============================================================
# BLOCK 11: Final Summary
# ============================================================

n_mendeley_train = len(train_idx)
n_mendeley_test  = len(test_idx)

print("\n" + "="*65)
print("  COMPLETE RESULTS SUMMARY")
print("="*65)

print(f"\n  MENDELEY WITHIN-DATASET (Enhanced Pipeline)")
print(f"  Train N={n_mendeley_train}  Test N={n_mendeley_test}")
print(f"  Accuracy : {acc_all*100:.2f}%  "
      f"95% CI [{acc_ci[0]*100:.2f}%, {acc_ci[1]*100:.2f}%]")
print(f"  Bal. Acc : {bacc_all*100:.2f}%  "
      f"95% CI [{bacc_ci[0]*100:.2f}%, {bacc_ci[1]*100:.2f}%]")
print(f"  Macro F1 : {mf1_all:.4f}  "
      f"95% CI [{mf1_ci[0]:.4f}, {mf1_ci[1]:.4f}]")
print(f"  MCC      : {mcc_all:.4f}")
print(f"  SNR>25dB : {acc25:.2f}%  MCC: {mcc25:.4f}  N={mask25.sum()}")
print(f"  Note: Mendeley lacks patient IDs; image-level independence assumed.")

print(f"\n  MULTI-COMPONENT PIPELINE COMPARISON (N={n_mendeley_test})")
for method_name, acc, mcc in [
    ("Baseline A: Argmin + fixed scale",  acc_a, mcc_a),
    ("Baseline B: Canny + skeleton",      acc_b, mcc_b),
    ("Final: Contour + grid + skeleton",  acc_f, mcc_f),
]:
    print(f"  {method_name:<40} Acc: {acc:.2f}%  MCC: {mcc:.4f}")

print(f"\n  DOMAIN SHIFT — ZERO-SHOT Analysis A (N=600)")
for code, d in results_a.items():
    ci = d['ci']
    print(f"  {code}: {d['name']:<28} "
          f"Acc={d['accuracy']*100:.2f}%  "
          f"BalAcc={d['balanced_accuracy']*100:.2f}%  "
          f"MCC={d['mcc']:.4f}")

print(f"\n  DOMAIN SHIFT — ZERO-SHOT Analysis B (fold-10, N={len(test_manifest)})")
for code, d in results_b.items():
    print(f"  {code}: {d['name']:<28} "
          f"Acc={d['accuracy']*100:.2f}%  "
          f"BalAcc={d['balanced_accuracy']*100:.2f}%  "
          f"MCC={d['mcc']:.4f}")

print(f"\n  DOMAIN SHIFT — DOMAIN-ADAPTED E7 "
      f"(train folds 1-8, val fold 9, test fold 10, N={len(test_manifest)})")
print(f"  Accuracy : {acc_e7*100:.2f}%  "
      f"95% CI [{e7_ci['acc'][0]*100:.2f}%, {e7_ci['acc'][1]*100:.2f}%]")
print(f"  Bal. Acc : {bacc_e7*100:.2f}%  "
      f"95% CI [{e7_ci['bacc'][0]*100:.2f}%, {e7_ci['bacc'][1]*100:.2f}%]")
print(f"  Macro F1 : {mf1_e7:.4f}  "
      f"95% CI [{e7_ci['mf1'][0]:.4f}, {e7_ci['mf1'][1]:.4f}]")
print(f"  MCC      : {mcc_e7:.4f}  "
      f"95% CI [{e7_ci['mcc'][0]:.4f}, {e7_ci['mcc'][1]:.4f}]")
print(f"  Epochs run: {len(history_rows)}  Best epoch: "
      f"{min(range(len(history_rows)), key=lambda i: history_rows[i]['val_loss']) + 1}")

print(f"\n  SAVED FIGURES ({FIGURES_DIR})")
fig_list = [
    ("Figure 3", "cm_final_enhanced_pipeline.png"),
    ("Figure 4", "confusion_matrix_snr25.png"),
    ("Figure 5", "ablation_comparison.png"),
    ("Figure 6", "degradation_experiment.png"),
    ("Figure 7", "domain_shift_analysis_a.png"),
    ("Figure 8", "domain_shift_analysis_b.png"),
    ("Supp.",    "cm_ptbxl_zero_shot_best.png"),
    ("Supp.",    "cm_ptbxl_domain_adapted.png"),
]
for tag, fname in fig_list:
    full   = FIGURES_DIR / fname
    status = "OK" if full.exists() else "NOT YET GENERATED"
    print(f"  {tag}: {fname}  [{status}]")

print(f"\n  SAVED RESULTS ({RESULTS_DIR})")
# Required files (4)
for fname in [
    'ptbxl_zero_shot_full.csv',
    'ptbxl_common_test.csv',
    'ptbxl_finetuning_history.csv',
    'split_summary.json',
]:
    full   = RESULTS_DIR / fname
    status = "OK" if full.exists() else "NOT YET GENERATED"
    print(f"  [required] {fname}  [{status}]")
# Additional files
for fname in [
    'mendeley_metrics.json',
    'mendeley_split_seed42.csv',
    'ptbxl_analysis_b_metrics.json',
    'ptbxl_e7_metrics.json',
]:
    full   = RESULTS_DIR / fname
    status = "OK" if full.exists() else "NOT YET GENERATED"
    print(f"  [additional] {fname}  [{status}]")

print("\n" + "="*65)